In [17]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,END,START,MessagesState
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

In [18]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")

In [19]:
def chat_node(state:MessagesState):
    response=llm.invoke(state["messages"])
    return{"messages":[response]}

In [20]:
builder = StateGraph(MessagesState)

In [21]:
builder.add_node("chat_node",chat_node)
builder.add_edge(START,"chat_node")
builder.add_edge("chat_node",END)
chatbot=builder.compile()

In [22]:
def run_cli_chatbot():
    print("Chatbot started! Type 'exit' or 'quit' to stop.")
    print()
    current_state = {"messages": []}
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ["exit", "quit"]:
            print("Exiting chat. Goodbye!")
            break

        if not user_input:
            continue

        current_state = chatbot.invoke(
            {"messages": current_state["messages"] + [HumanMessage(content=user_input)]}
        )
        latest_ai_message = current_state["messages"][-1]
        print(f"AI: {latest_ai_message.content}")
        print(f"[State Tracker] Total messages in list: {len(current_state['messages'])}")
        for idx, msg in enumerate(current_state["messages"], start=1):
            msg_type = type(msg).__name__
            preview = msg.content[:40].replace("\n", " ")
            print(f"  {idx}. [{msg_type}] -> {preview}...")
        print("=" * 50)
        print()


In [ ]:
if __name__ == "__main__":
    run_cli_chatbot()